## FROM SCRATCH

In [1]:
import numpy as np

# Features:
# Weather:     Sunny=0, Cloudy=1, Rainy=2
# Temperature: Hot=0, Mild=1, Cool=2
# Wind:        Weak=0, Strong=1

X = np.array([
    [0, 0, 0],  # Sunny, Hot, Weak
    [0, 0, 1],  # Sunny, Hot, Strong
    [1, 0, 0],  # Cloudy, Hot, Weak
    [2, 1, 0],  # Rainy, Mild, Weak
    [2, 2, 1],  # Rainy, Cool, Strong
    [1, 1, 1],  # Cloudy, Mild, Strong
    [0, 1, 0],  # Sunny, Mild, Weak
    [2, 0, 0],  # Rainy, Hot, Weak
])

# No=0, Yes=1
y = np.array([0, 0, 1, 1, 0, 1, 1, 1])

# Number of possible categories in each feature
n_categories = [3, 3, 2]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (8, 3)
y shape: (8,)


In [2]:
class CategoricalNB:
    def __init__(self, alpha=1.0, n_categories=None):
        self.alpha = alpha
        self.n_categories = n_categories

    def fit(self, X, y):
        X = np.asarray(X, dtype=int)
        y = np.asarray(y)

        if self.alpha <= 0:
            raise ValueError("Alpha must be positive.")

        if np.any(X < 0):
            raise ValueError("Category codes must be non-negative.")

        self.classes_ = np.unique(y)

        n_samples, n_features = X.shape
        n_classes = len(self.classes_)

        if self.n_categories is None:
            self.n_categories_ = [
                int(X[:, j].max()) + 1
                for j in range(n_features)
            ]
        else:
            self.n_categories_ = list(self.n_categories)

        if len(self.n_categories_) != n_features:
            raise ValueError(
                "n_categories must contain one value per feature."
            )

        self.class_count_ = np.zeros(n_classes)
        self.class_prior_ = np.zeros(n_classes)

        # Counts for every class, feature and category
        self.feature_count_ = [
            np.zeros((n_classes, n_categories))
            for n_categories in self.n_categories_
        ]

        for class_index, class_label in enumerate(self.classes_):

            X_class = X[y == class_label]

            self.class_count_[class_index] = len(X_class)

            self.class_prior_[class_index] = (
                len(X_class) / n_samples
            )

            for feature_index, n_categories in enumerate(
                self.n_categories_
            ):

                for category in range(n_categories):

                    count = np.sum(
                        X_class[:, feature_index] == category
                    )

                    self.feature_count_[feature_index][
                        class_index, category
                    ] = count

        # Calculate smoothed log probabilities
        self.feature_log_prob_ = []

        for feature_index, n_categories in enumerate(
            self.n_categories_
        ):

            counts = self.feature_count_[feature_index]

            smoothed_counts = counts + self.alpha

            denominators = (
                self.class_count_[:, None]
                + self.alpha * n_categories
            )

            log_probabilities = np.log(
                smoothed_counts / denominators
            )

            self.feature_log_prob_.append(
                log_probabilities
            )

        return self

    def predict_log_proba(self, X):
        X = np.asarray(X, dtype=int)

        log_prior = np.log(self.class_prior_)

        log_scores = np.tile(
            log_prior,
            (len(X), 1)
        )

        for feature_index in range(X.shape[1]):

            categories = X[:, feature_index]

            log_scores += self.feature_log_prob_[
                feature_index
            ][:, categories].T

        return log_scores

    def predict(self, X):
        log_scores = self.predict_log_proba(X)

        predicted_indices = np.argmax(
            log_scores,
            axis=1
        )

        return self.classes_[predicted_indices]

    def predict_proba(self, X):
        log_scores = self.predict_log_proba(X)

        shifted_scores = (
            log_scores
            - np.max(log_scores, axis=1, keepdims=True)
        )

        exp_scores = np.exp(shifted_scores)

        return exp_scores / np.sum(
            exp_scores,
            axis=1,
            keepdims=True
        )

In [3]:
model = CategoricalNB(
    alpha=1.0,
    n_categories=n_categories
)

model.fit(X, y)

print("Classes:", model.classes_)
print("Class priors:", model.class_prior_)

print("\nWeather counts:")
print(model.feature_count_[0])

print("\nTemperature counts:")
print(model.feature_count_[1])

print("\nWind counts:")
print(model.feature_count_[2])

Classes: [0 1]
Class priors: [0.375 0.625]

Weather counts:
[[2. 0. 1.]
 [1. 2. 2.]]

Temperature counts:
[[2. 0. 1.]
 [2. 3. 0.]]

Wind counts:
[[1. 2.]
 [4. 1.]]


In [4]:
new_observation = np.array([
    [1, 1, 0]
])

prediction = model.predict(new_observation)

probabilities = model.predict_proba(new_observation)

print("Prediction:", prediction[0])
print("Class order:", model.classes_)
print("Probabilities:", probabilities)

Prediction: 1
Class order: [0 1]
Probabilities: [[0.04741744 0.95258256]]


## SCIKIT LEARN

In [5]:
from sklearn.naive_bayes import CategoricalNB as SklearnCategoricalNB

sklearn_model = SklearnCategoricalNB(
    alpha=1.0,
    min_categories=n_categories
)

sklearn_model.fit(X, y)

sklearn_prediction = sklearn_model.predict(
    new_observation
)

sklearn_probabilities = sklearn_model.predict_proba(
    new_observation
)

print("Scratch prediction:", prediction)
print("Sklearn prediction:", sklearn_prediction)

print("\nScratch probabilities:", probabilities)
print("Sklearn probabilities:", sklearn_probabilities)

print(
    "\nPredictions match:",
    np.array_equal(prediction, sklearn_prediction)
)

print(
    "Probabilities match:",
    np.allclose(probabilities, sklearn_probabilities)
)

Scratch prediction: [1]
Sklearn prediction: [1]

Scratch probabilities: [[0.04741744 0.95258256]]
Sklearn probabilities: [[0.04741744 0.95258256]]

Predictions match: True
Probabilities match: True


## REAL WORLD DATASET

In [6]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo

# Fetch the Mushroom dataset from UCI
mushroom = fetch_ucirepo(id=73)

X = mushroom.data.features.copy()
y = mushroom.data.targets.iloc[:, 0].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.value_counts())

Feature shape: (8124, 22)
Target shape: (8124,)


,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,x,s,n,t,p,f,c,n,k,e,...,s,w,w,p,w,o,p,k,s,u
1,x,s,y,t,a,f,c,b,k,e,...,s,w,w,p,w,o,p,n,n,g
2,b,s,w,t,l,f,c,b,n,e,...,s,w,w,p,w,o,p,n,n,m
3,x,y,w,t,p,f,c,n,n,e,...,s,w,w,p,w,o,p,k,s,u
4,x,s,g,f,n,f,w,b,k,t,...,s,w,w,p,w,o,e,n,a,g


poisonous
e    4208
p    3916
Name: count, dtype: int64

In [7]:
# Check column names and data types
print(X.dtypes.value_counts())

# Count missing values represented as '?'
missing_counts = (X == "?").sum().sort_values(ascending=False)

display(missing_counts[missing_counts > 0])

# Check target distribution
print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

str    22
Name: count, dtype: int64


Series([], dtype: int64)

poisonous
e    4208
p    3916
Name: count, dtype: int64
poisonous
e    0.518
p    0.482
Name: proportion, dtype: float64


In [8]:
for column in X.columns:
    print(column, X[column].map(type).value_counts().to_dict())

cap-shape {<class 'str'>: 8124}
cap-surface {<class 'str'>: 8124}
cap-color {<class 'str'>: 8124}
bruises {<class 'str'>: 8124}
odor {<class 'str'>: 8124}
gill-attachment {<class 'str'>: 8124}
gill-spacing {<class 'str'>: 8124}
gill-size {<class 'str'>: 8124}
gill-color {<class 'str'>: 8124}
stalk-shape {<class 'str'>: 8124}
stalk-root {<class 'str'>: 5644, <class 'float'>: 2480}
stalk-surface-above-ring {<class 'str'>: 8124}
stalk-surface-below-ring {<class 'str'>: 8124}
stalk-color-above-ring {<class 'str'>: 8124}
stalk-color-below-ring {<class 'str'>: 8124}
veil-type {<class 'str'>: 8124}
veil-color {<class 'str'>: 8124}
ring-number {<class 'str'>: 8124}
ring-type {<class 'str'>: 8124}
spore-print-color {<class 'str'>: 8124}
population {<class 'str'>: 8124}
habitat {<class 'str'>: 8124}


In [9]:
# Store category mappings and category counts
category_maps = {}
n_categories = []

X_encoded = pd.DataFrame(index=X.index)

for column in X.columns:
    # Force every value to a Python string
    values = [str(value) for value in X[column].tolist()]

    # Get unique categories without sorting
    categories = list(dict.fromkeys(values))

    # Create category -> integer mapping
    category_maps[column] = {
        category: code
        for code, category in enumerate(categories)
    }

    # Encode feature
    X_encoded[column] = [
        category_maps[column][value]
        for value in values
    ]

    n_categories.append(len(categories))

# Convert to NumPy integer array
X_encoded = X_encoded.to_numpy(dtype=np.int64)

# Encode target: edible=0, poisonous=1
y_values = [str(value) for value in y.tolist()]
y_encoded = np.array(
    [{"e": 0, "p": 1}[value] for value in y_values],
    dtype=np.int64,
)

print("Encoded X shape:", X_encoded.shape)
print("Encoded y shape:", y_encoded.shape)
print("Category counts:", n_categories)

assert np.all(X_encoded >= 0)
assert set(np.unique(y_encoded)) == {0, 1}

Encoded X shape: (8124, 22)
Encoded y shape: (8124,)
Category counts: [6, 4, 10, 2, 9, 2, 2, 2, 12, 2, 5, 4, 4, 9, 9, 1, 4, 3, 5, 9, 6, 7]


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded,
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("Train class proportions:")
print(pd.Series(y_train).value_counts(normalize=True).sort_index())

print("Test class proportions:")
print(pd.Series(y_test).value_counts(normalize=True).sort_index())

Train: (6499, 22)
Test: (1625, 22)
Train class proportions:
0    0.517926
1    0.482074
Name: proportion, dtype: float64
Test class proportions:
0    0.518154
1    0.481846
Name: proportion, dtype: float64


In [11]:
# Our scratch implementation
scratch_model = CategoricalNB(
    alpha=1.0,
    n_categories=n_categories,
)

scratch_model.fit(X_train, y_train)

scratch_predictions = scratch_model.predict(X_test)
scratch_probabilities = scratch_model.predict_proba(X_test)

In [12]:
from sklearn.naive_bayes import CategoricalNB as SklearnCategoricalNB

sklearn_model = SklearnCategoricalNB(
    alpha=1.0,
    min_categories=n_categories,
)

sklearn_model.fit(X_train, y_train)

sklearn_predictions = sklearn_model.predict(X_test)
sklearn_probabilities = sklearn_model.predict_proba(X_test)

In [13]:
print(
    "Predictions identical:",
    np.array_equal(scratch_predictions, sklearn_predictions)
)

print(
    "Probabilities close:",
    np.allclose(
        scratch_probabilities,
        sklearn_probabilities,
        atol=1e-10,
    )
)

print("\nScratch probabilities:")
print(scratch_probabilities[:5])

print("\nSklearn probabilities:")
print(sklearn_probabilities[:5])

Predictions identical: True
Probabilities close: True

Scratch probabilities:
[[2.93934194e-12 1.00000000e+00]
 [6.30768682e-01 3.69231318e-01]
 [9.99998306e-01 1.69422230e-06]
 [6.02296162e-12 1.00000000e+00]
 [1.49607586e-12 1.00000000e+00]]

Sklearn probabilities:
[[2.93934194e-12 1.00000000e+00]
 [6.30768682e-01 3.69231318e-01]
 [9.99998306e-01 1.69422230e-06]
 [6.02296162e-12 1.00000000e+00]
 [1.49607586e-12 1.00000000e+00]]


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

print("Accuracy :", accuracy_score(y_test, scratch_predictions))
print("Precision:", precision_score(y_test, scratch_predictions))
print("Recall   :", recall_score(y_test, scratch_predictions))
print("F1       :", f1_score(y_test, scratch_predictions))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, scratch_predictions))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        scratch_predictions,
        target_names=["edible", "poisonous"],
    )
)

Accuracy : 0.9458461538461539
Precision: 0.9901269393511989
Recall   : 0.896551724137931
F1       : 0.9410187667560321

Confusion matrix:
[[835   7]
 [ 81 702]]

Classification report:
              precision    recall  f1-score   support

      edible       0.91      0.99      0.95       842
   poisonous       0.99      0.90      0.94       783

    accuracy                           0.95      1625
   macro avg       0.95      0.94      0.95      1625
weighted avg       0.95      0.95      0.95      1625



In [15]:
# Select one test sample
i = 0

# Show original categorical values
sample_original = X.iloc[
    X.index.get_indexer([X.index[X.index.isin(X.index)][0]])  # placeholder removed below
]

In [16]:
# Preserve original row indices during the split
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded,
)

sample_raw = X_test_raw.iloc[[i]]
sample_encoded = X_test[i:i+1]

actual_label = y_test[i]
predicted_label = scratch_model.predict(sample_encoded)[0]
predicted_probs = scratch_model.predict_proba(sample_encoded)[0]

display(sample_raw.T)

print("Actual:", "edible" if actual_label == 0 else "poisonous")
print("Predicted:", "edible" if predicted_label == 0 else "poisonous")
print("P(edible):", predicted_probs[0])
print("P(poisonous):", predicted_probs[1])

,4632
cap-shape,x
cap-surface,y
cap-color,g
bruises,f
odor,f
gill-attachment,f
gill-spacing,c
gill-size,b
gill-color,p
stalk-shape,e


Actual: poisonous
Predicted: poisonous
P(edible): 2.939341944509238e-12
P(poisonous): 0.9999999999970606


In [17]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

alpha_results = []

for alpha in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]:
    model = SklearnCategoricalNB(
        alpha=alpha,
        min_categories=n_categories,
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
    )

    alpha_results.append({
        "alpha": alpha,
        "mean_f1": scores.mean(),
        "std_f1": scores.std(),
    })

alpha_results = pd.DataFrame(alpha_results)
display(alpha_results.sort_values("mean_f1", ascending=False))

,alpha,mean_f1,std_f1
0,0.01,0.990382,0.002542
1,0.10,0.975752,0.005922
2,0.50,0.953413,0.009711
3,1.00,0.944811,0.012439
4,2.00,0.940023,0.012435
5,5.00,0.932951,0.012622


In [18]:
# Select alpha based on training cross-validation
best_alpha = 0.01

final_model = CategoricalNB(
    alpha=best_alpha,
    n_categories=n_categories,
)

final_model.fit(X_train, y_train)

final_predictions = final_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, final_predictions))
print(
    classification_report(
        y_test,
        final_predictions,
        target_names=["edible", "poisonous"],
    )
)

print("Confusion matrix:")
print(confusion_matrix(y_test, final_predictions))

Test accuracy: 0.9895384615384616
              precision    recall  f1-score   support

      edible       0.99      0.99      0.99       842
   poisonous       0.99      0.99      0.99       783

    accuracy                           0.99      1625
   macro avg       0.99      0.99      0.99      1625
weighted avg       0.99      0.99      0.99      1625

Confusion matrix:
[[835   7]
 [ 10 773]]


## Conclusion: Categorical Naive Bayes

Implemented Categorical Naive Bayes from scratch using NumPy and validated it against scikit-learn's `CategoricalNB`.

The model was evaluated on the UCI Mushroom dataset, containing 8,124 samples and 22 categorical features. The features were encoded as non-negative integer category IDs. Missing stalk-root values represented by `?` were retained as a separate category.

With `alpha=1.0`, the scratch implementation produced predictions and probabilities matching scikit-learn. On the held-out test set, it achieved 94.58% accuracy and 94.10% F1 for the poisonous class.

Cross-validation on the training set selected `alpha=0.01` as the best tested smoothing value. The tuned model was then evaluated on the held-out test set.

This experiment demonstrated categorical likelihood estimation, class priors, Laplace smoothing, log-probability calculations, category encoding, and model evaluation.

The model assumes conditional independence between features given the class. Its predictions and probabilities must not be treated as reliable mushroom safety assessments.
